# YouTube Title Predictor

**Problem.** Small travel channels like my channel, KarmicSteps (~340 subscribers), often have
2–3 title ideas for a video and no data on which will perform better. This project predicts,
from information known at upload, whether a long-form travel video will **beat its own
channel's recent typical views**. The model is used to **rank candidate titles** before publishing.

**Data.** Videos from small travel channels (1k–100k subscribers) collected with the
YouTube Data API on 2026-10-26 (`collect_data.py`, channel selection in `explore_api.ipynb`).

**Target.** `beat_baseline = 1` if a video's views exceed the median of its channel's
previous 20 long-form videos.

**Metric.** ROC AUC, with cross-validation grouped by channel (the model is always scored on
channels it has never seen, which is the situation when scoring KarmicSteps titles).


## 1. Data cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


: 

### 1.1 Load the data and fix the types
`COLLECTION_DATE` is fixed, so the age filter gives the same result on every run.
Empty tags and descriptions are read as NaN, so they're filled with empty text.

In [ ]:
COLLECTION_DATE = pd.Timestamp("2026-10-26", tz="UTC")

videos = pd.read_csv("data/videos.csv")
print(videos.shape)
videos.head()

In [ ]:
videos["tags"] = videos["tags"].fillna("")
videos["description"] = videos["description"].fillna("")
videos["published_at"] = pd.to_datetime(videos["published_at"])
videos["duration_s"] = videos["duration"].apply(lambda d: pd.Timedelta(d).total_seconds())
videos["age_days"] = (COLLECTION_DATE - videos["published_at"]).dt.days
videos.dtypes

### 1.2 Long-form only, no live streams
Shorts can be up to 3 minutes, so only videos longer than 180 s are kept. Past live streams
are removed too: their views behave differently from normal uploads.

In [ ]:
videos = videos[videos['duration_s'] > 180]
print("After filtering long-form videos:", len(videos))
videos = videos[~videos["was_live"]]
print("After removing live streams:", len(videos))

## 2. Target: beat the channel's recent median
Each video is compared with the median views of its own channel's previous 20 long-form
videos. This removes the effect of channel size and uses only information available at upload.

### 2.1 Rolling median of the previous 20 videos
`shift(1)` makes sure a video's own views are never part of its baseline (no leakage).
A channel's first 20 videos have no full baseline (`NaN`), so they're dropped in 2.2.

In [ ]:
videos = videos.sort_values(["channel_id", "published_at"])         # channel first, then publish date

videos["baseline"] = (
    videos.groupby("channel_id")["view_count"]
    .transform(lambda s: s.shift(1).rolling(20, min_periods=20).median())
)

videos["beat_baseline"] = (videos["view_count"] > videos["baseline"]).astype(int)

print("Rows with a baseline:", videos["baseline"].notna().sum())
videos[["channel_id", "published_at", "view_count", "baseline", "beat_baseline"]].head(25)


### 2.2 Drop rows without a baseline, apply the age window
Only videos aged **90 days to 3 years** are used as examples.

- The minimum was raised from 30 to 90 days after EDA (section 3.3): videos aged 30–90 days beat their baseline 65% of the time (median 1.4× the baseline), while every older age band is close to 50%. This is most likely a selection effect: the channels were found through search in October 2026, which favours channels that are growing right now.
- Videos older than 3 years may reflect what worked years ago.

Older videos are still used as *history* in the baselines, which is why this filter comes after 2.1.

In [ ]:
df = videos[videos["baseline"].notna()]
print("With a baseline:", len(df))

df = df[(df["age_days"] >= 90) & (df["age_days"] <= (3 * 365))]
print("With age between 90 days and 3 years:", len(df))

print("Channels:", df["channel_id"].nunique())
df["beat_baseline"].value_counts(normalize=True)

In [ ]:
per_channel = df["channel_id"].value_counts()
print(per_channel.describe())
print("Channels with < 20 videos:", (per_channel < 20).sum())

### 2.3 Language filter and minimum videos per channel

In [ ]:
def ascii_share(title):
    letters = [c for c in title if c.isalpha()]
    if not letters:
        return 1.0
    return sum(c.isascii() for c in letters) / len(letters)

df["ascii_share"] = df["title"].apply(ascii_share)
print(df[df["ascii_share"] < 0.9]["title"].head())

Channels where more than 10% of titles are in a non-Latin script (Hindi, Punjabi, Tamil, Kannada, Korean) are dropped entirely, keeping channels that publish mainly in English, like KarmicSteps. Remaining individual non-English titles are dropped too. Known limitation: mixed Hindi–English titles and Latin-script languages (e.g. Turkish) can still slip through.

In [ ]:
# 1. Channel-level: share of each channel's titles that look non-English
df["non_english"] = df["ascii_share"] < 0.9
channel_share = df.groupby("channel_id")["non_english"].mean()
english_channels = channel_share[channel_share <= 0.1].index
df = df[df["channel_id"].isin(english_channels)]
print("After dropping non-English channels:", len(df), "videos,", df["channel_id"].nunique(), "channels")

# 2. Title-level: drop remaining non-English titles
df = df[~df["non_english"]]
print("After dropping non-English titles:", len(df))

# 3. Per-channel floor
counts = df["channel_id"].value_counts()
keep = counts[counts >= 20].index
df = df[df["channel_id"].isin(keep)]
print("After the 20-video floor:", len(df), "videos,", df["channel_id"].nunique(), "channels")

df["beat_baseline"].value_counts(normalize=True)


### 2.4 Save the clean dataset
`train.py` loads this file, so the cleaning steps don't have to be repeated.

In [ ]:
df.to_csv("data/videos_clean.csv", index=False)

## 3. Exploratory data analysis

### 3.1 Target balance

In [ ]:
plt.figure(figsize=(8, 6))
df['beat_baseline'].value_counts().sort_index().plot(kind='bar')
plt.xticks([0, 1], ['Below baseline (0)', 'Beat baseline (1)'], rotation=0)
plt.ylabel('Videos')
plt.title('Target balance: beat the channel baseline?')
plt.show()

### 3.2 Videos per channel
The 29 channels contribute between 22 and 272 videos each. The four largest channels hold about 35% of all videos, so they have more weight in training, but no single channel dominates.

In [ ]:
channel_names = pd.read_csv("data/channels.csv").set_index("channel_id")["title"]
per_channel = df["channel_id"].map(channel_names).value_counts()

per_channel.plot(kind="bar", figsize=(12, 4), title="Videos per channel")
plt.ylabel("Videos")
plt.show()

### 3.3 Does the target depend on video age?
With the original 30-day minimum, the newest videos behaved differently from the rest:

| Age band | Videos | Beat baseline | Median views ÷ baseline |
|---|---|---|---|
| 30–90 days | 192 | 65% | 1.40× |
| 90–180 days | 285 | 47% | 0.98× |
| 180–365 days | 604 | 50% | 1.00× |
| 1–2 years | 996 | 49% | 0.98× |
| 2–3 years | 885 | 51% | 1.01× |

So the minimum age was raised to 90 days (see 2.2). After the change, every age band is close to 50%:

In [ ]:
df["age_band"] = pd.cut(df["age_days"], bins=[90, 180, 365, 730, 1095])
df.groupby("age_band")['beat_baseline'].mean()

### 3.4 Feature importance: numerical and categorical features

Same approach as the churn project in module 3:
- **Numerical features:** correlation with the target.
- **Categorical features:** the beat rate per group compared with the global rate (`diff` and `risk`),
  and mutual information.

All features are known at upload time.


In [ ]:
# Create the features from the raw columns
df['published_at'] = pd.to_datetime(df['published_at'])
df['duration_min'] = df['duration_s'] / 60
df['publish_hour'] = df['published_at'].dt.hour          # UTC hour
df['publish_weekday'] = df['published_at'].dt.day_name()
df['n_tags'] = df['tags'].apply(lambda t: len(t.split('|')) if t else 0)
df['description_length'] = df['description'].str.len()
df['title_length'] = df['title'].str.len()

numerical = ['duration_min', 'publish_hour', 'n_tags', 'description_length', 'title_length']
categorical = ['publish_weekday']

In [ ]:
# Correlation of each numerical feature with the target
df[numerical].corrwith(df.beat_baseline)

**Result:** All correlations with the target are small (|r| < 0.05). The strongest is duration
(r = 0.05: longer videos beat their baseline slightly more often). Publish hour, tag count,
description length and title length are close to 0, so on their own these features carry little signal.

In [ ]:
# Beat rate per weekday, compared with the global rate
global_rate = df.beat_baseline.mean()

df_group = df.groupby('publish_weekday').beat_baseline.agg(['mean', 'count'])
df_group['diff'] = df_group['mean'] - global_rate
df_group['risk'] = df_group['mean'] / global_rate
df_group.sort_values('risk')

In [ ]:
# Mutual information between the categorical feature and the target
from sklearn.metrics import mutual_info_score

mutual_info_score(df.beat_baseline, df.publish_weekday)


**Result:** Monday and Tuesday uploads beat their baseline slightly more often (~54%, risk ≈ 1.09)
than Saturday and Wednesday uploads (~47%, risk ≈ 0.95). The mutual information is low, so weekday
is a weak signal. It also can't change the ranking of candidate titles, because all candidates share the same upload day.


### 3.5 Title words: most common words for beat vs not beat
The use case only varies the title, so the words themselves need to carry signal.
Each title is split into words, and the same `diff` / `risk` analysis is applied per word:
of all titles containing a word, what share beat their baseline?

In [ ]:
def words(title):
    # Split a title into lowercase words and strip punctuation from the edges
    return [w.strip('.,!?|:()-–[]"\'').lower() for w in title.split()]

# One row per (word, title): each word counted once per title, stop words and empty strings skipped
rows = []
for title, target in zip(df.title, df.beat_baseline):
    for w in set(words(title)):
        if w and w not in ENGLISH_STOP_WORDS:
            rows.append({'word': w, 'beat_baseline': target})

df_words = pd.DataFrame(rows)

# Beat rate per word compared with the global rate (same idea as the churn risk ratios in module 3)
global_rate = df.beat_baseline.mean()

df_group = df_words.groupby('word').beat_baseline.agg(['mean', 'count'])
df_group['diff'] = df_group['mean'] - global_rate
df_group['risk'] = df_group['mean'] / global_rate

# Only words that appear in at least 30 titles, so the rates aren't based on a handful of videos
df_group = df_group[df_group['count'] >= 30].sort_values('risk')

print('Words that lean towards NOT beating the baseline:')
display(df_group.head(15))
print('Words that lean towards beating the baseline:')
display(df_group.tail(15))

**Result:** Some words clearly lean one way (overall beat rate ≈ 50%):
- **Towards beating the baseline:** concrete experiences and value words such as *hotel* (69%),
  *breakfast* (68%), *eat* (68%), *shopping* (64%), *new* (63%) and *worth* (63%, as in "is it worth it?").
- **Towards not beating it:** generic words such as *trip* (38%), *things* (34%), *places* (37%),
  *tips* (37%), *walk* (35%), *life* (39%) and *episode* (40%).

Caution: some strong words come from a single channel's series (e.g. *bar* 87%, *scholarship*,
*global*, *volleyball*), so they may reflect that channel rather than a general title effect.
Cross-validation grouped by channel (section 5) will show whether the title signal generalises
to channels the model hasn't seen. Overall, the words do carry signal, which supports using TF-IDF on titles.
